# MHEALTH — Data Prep + Centralized LSTM Baseline

This notebook:
1. Loads the MHEALTH dataset (10 subjects, wearable sensor logs)
2. Windows each subject's signal into fixed-length segments (per-client FL data)
3. Trains a centralized LSTM baseline (no FL, no DP, no HE) to get a ceiling accuracy

This is the reference point your full hybrid FL+DP+HE framework will be compared against.

## 0. Setup

Upload `mhealth_dataset.zip` (the UCI MHEALTH dataset) using the file browser on the left, or via the upload cell below.

In [1]:
# If running in Colab, uncomment to upload the zip interactively:
# from google.colab import files
# uploaded = files.upload()  # select mhealth_dataset.zip

In [1]:
import zipfile, os

ZIP_PATH = "mhealth+dataset.zip"  # adjust path if needed   # adjust path if needed
EXTRACT_DIR = "mhealth"

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)

DATA_DIR = os.path.join(EXTRACT_DIR, "MHEALTHDATASET")
print(os.listdir(DATA_DIR))

['mHealth_subject1.log', 'mHealth_subject10.log', 'mHealth_subject2.log', 'mHealth_subject3.log', 'mHealth_subject4.log', 'mHealth_subject5.log', 'mHealth_subject6.log', 'mHealth_subject7.log', 'mHealth_subject8.log', 'mHealth_subject9.log', 'README.txt']


In [2]:
import sys
!{sys.executable} -m pip install -q torch scikit-learn pandas numpy

'c:\Users\joank\OneDrive' is not recognized as an internal or external command,
operable program or batch file.


## 1. Windowing / preprocessing

In [3]:
!pip install numpy pandas

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

SUBJECTS = [f"mHealth_subject{i}.log" for i in range(1, 11)]

FS = 50  # Hz
WINDOW_SEC = 2.5
OVERLAP = 0.5  # 50% overlap
WINDOW_SIZE = int(FS * WINDOW_SEC)          # 125 samples
STEP = int(WINDOW_SIZE * (1 - OVERLAP))     # 62 samples (~50% overlap)

N_CHANNELS = 23  # columns 1-23 are sensor features, column 24 is label

ACTIVITY_NAMES = {
    0: "Null", 1: "Standing", 2: "Sitting", 3: "Lying", 4: "Walking",
    5: "Climbing stairs", 6: "Waist bends", 7: "Arms elevation",
    8: "Knees bending", 9: "Cycling", 10: "Jogging", 11: "Running",
    12: "Jump front&back",
}


def load_subject(path):
    df = pd.read_csv(path, sep=r"\s+", header=None)
    X = df.iloc[:, :N_CHANNELS].values.astype(np.float32)
    y = df.iloc[:, N_CHANNELS].values.astype(np.int32)
    return X, y


def window_signal(X, y, window_size=WINDOW_SIZE, step=STEP, drop_null=True, purity_thresh=0.9):
    """
    Slide a window over X/y. Keep a window only if a single label
    dominates >= purity_thresh of the window (avoids boundary-blur
    between activities). Label = majority label in the window.
    """
    n_samples = X.shape[0]
    windows, labels = [], []

    for start in range(0, n_samples - window_size + 1, step):
        end = start + window_size
        wx = X[start:end]
        wy = y[start:end]

        vals, counts = np.unique(wy, return_counts=True)
        majority_label = vals[np.argmax(counts)]
        purity = counts.max() / window_size

        if drop_null and majority_label == 0:
            continue
        if purity < purity_thresh:
            continue

        windows.append(wx)
        labels.append(majority_label)

    if len(windows) == 0:
        return np.empty((0, window_size, N_CHANNELS), dtype=np.float32), np.empty((0,), dtype=np.int32)

    return np.stack(windows), np.array(labels, dtype=np.int32)


def build_client_datasets(data_dir):
    client_data = {}
    summary_rows = []

    for fname in SUBJECTS:
        subject_id = fname.replace("mHealth_subject", "").replace(".log", "")
        path = Path(data_dir) / fname
        X, y = load_subject(path)
        Xw, yw = window_signal(X, y)

        client_data[subject_id] = (Xw, yw)

        vals, counts = np.unique(yw, return_counts=True)
        dist = {int(v): int(c) for v, c in zip(vals, counts)}
        summary_rows.append({
            "subject": subject_id,
            "raw_samples": X.shape[0],
            "n_windows": Xw.shape[0],
            "n_classes_present": len(vals),
            **{f"L{v}": dist.get(v, 0) for v in range(1, 13)},
        })

    summary_df = pd.DataFrame(summary_rows).set_index("subject")
    return client_data, summary_df

In [5]:
client_data, summary_df = build_client_datasets(DATA_DIR)

print(f"Window size: {WINDOW_SIZE} samples ({WINDOW_SEC}s @ {FS}Hz), step: {STEP} samples ({OVERLAP*100:.0f}% overlap)\n")
print("=== Per-client (per-subject) window summary ===")
print(summary_df[["raw_samples", "n_windows", "n_classes_present"]].to_string())

print("\n=== Total windows across all clients ===")
print(summary_df["n_windows"].sum())

Window size: 125 samples (2.5s @ 50Hz), step: 62 samples (50% overlap)

=== Per-client (per-subject) window summary ===
         raw_samples  n_windows  n_classes_present
subject                                           
1             161280        548                 12
2             130561        552                 12
3             122112        548                 12
4             116736        549                 12
5             119808        529                 12
6              98304        501                 12
7             104448        527                 12
8             129024        517                 12
9             135168        536                 12
10             98304        524                 12

=== Total windows across all clients ===
5331


In [6]:
print("=== Per-class window counts (aggregated across all clients) ===")
class_cols = [c for c in summary_df.columns if c.startswith("L")]
class_totals = summary_df[class_cols].sum().sort_index(key=lambda x: [int(s[1:]) for s in x])
for col, total in class_totals.items():
    cid = int(col[1:])
    print(f"  {col} ({ACTIVITY_NAMES[cid]}): {total}")

print("\n=== Example tensor shape for one client (subject 1) ===")
Xw, yw = client_data["1"]
print(f"  X shape: {Xw.shape}  (n_windows, window_size, n_channels)")
print(f"  y shape: {yw.shape}")

=== Per-class window counts (aggregated across all clients) ===
  L1 (Standing): 479
  L2 (Sitting): 480
  L3 (Lying): 478
  L4 (Walking): 480
  L5 (Climbing stairs): 469
  L6 (Waist bends): 441
  L7 (Arms elevation): 459
  L8 (Knees bending): 456
  L9 (Cycling): 480
  L10 (Jogging): 479
  L11 (Running): 479
  L12 (Jump front&back): 151

=== Example tensor shape for one client (subject 1) ===
  X shape: (548, 125, 23)  (n_windows, window_size, n_channels)
  y shape: (548,)


In [9]:
import sys
print(sys.executable)


c:\Users\joank\OneDrive - Lancaster University\Desktop\Richard's Project\.venv\.venv\.venv\Scripts\python.exe


## 2. Centralized LSTM baseline



In [7]:
import sys
!{sys.executable} -m pip install -q torch scikit-learn pandas numpy

'c:\Users\joank\OneDrive' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def pool_clients(client_data):
    Xs, ys, subject_ids = [], [], []
    for subj, (Xw, yw) in client_data.items():
        Xs.append(Xw)
        ys.append(yw)
        subject_ids.extend([subj] * len(yw))
    X = np.concatenate(Xs, axis=0)
    y = np.concatenate(ys, axis=0)
    return X, y, np.array(subject_ids)


class LSTMClassifier(nn.Module):
    def __init__(self, n_channels, n_classes, hidden1=64, hidden2=32, dense=32, dropout=0.3):
        super().__init__()
        self.lstm1 = nn.LSTM(n_channels, hidden1, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True)
        self.drop2 = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden2, dense)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(dense, n_classes)
        # NOTE: unlike the Keras version this does not mask zero-padded
        # timesteps; MHEALTH windows here are fixed-length so masking
        # isn't needed. Add nn.utils.rnn packing if you introduce padding.

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.drop1(out)
        out, _ = self.lstm2(out)
        out = out[:, -1, :]  # last timestep
        out = self.drop2(out)
        out = self.relu(self.fc1(out))
        out = self.fc2(out)  # raw logits; use CrossEntropyLoss (applies softmax internally)
        return out


def build_lstm_model(n_channels, n_classes):
    return LSTMClassifier(n_channels, n_classes).to(device)

ModuleNotFoundError: No module named 'torch'

In [7]:
X, y, subject_ids = pool_clients(client_data)

# labels are 1-12 -> remap to 0-11 for sparse_categorical_crossentropy
label_map = {orig: idx for idx, orig in enumerate(sorted(np.unique(y)))}
inv_label_map = {v: k for k, v in label_map.items()}
y_mapped = np.array([label_map[v] for v in y])
n_classes = len(label_map)

print(f"Pooled dataset: X={X.shape}, y={y_mapped.shape}, n_classes={n_classes}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_mapped, test_size=0.2, stratify=y_mapped, random_state=SEED
)

# normalize per-channel using train stats only (avoid leakage)
n_train, win, ch = X_train.shape
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, ch))

def scale(X):
    shape = X.shape
    return scaler.transform(X.reshape(-1, ch)).reshape(shape).astype(np.float32)

X_train_s = scale(X_train)
X_test_s = scale(X_test)

Pooled dataset: X=(5331, 125, 23), y=(5331,), n_classes=12


In [8]:
model = build_lstm_model(n_channels=ch, n_classes=n_classes)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable params: {n_params}")

LSTMClassifier(
  (lstm1): LSTM(23, 64, batch_first=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (lstm2): LSTM(64, 32, batch_first=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=32, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=12, bias=True)
)

Total trainable params: 36780


In [9]:
import copy
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 32
EPOCHS = 60
PATIENCE = 8
VAL_SPLIT = 0.15

# carve out a validation split from the training set (mirrors validation_split in Keras)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_s, y_train, test_size=VAL_SPLIT, stratify=y_train, random_state=SEED
)

train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr).long())
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val).long())
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val_acc = 0.0
best_state = None
epochs_no_improve = 0
history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += xb.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * xb.size(0)
            val_correct += (out.argmax(1) == yb).sum().item()
            val_total += xb.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    history["loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["accuracy"].append(train_acc)
    history["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch+1:3d}/{EPOCHS} - loss: {train_loss:.4f} - acc: {train_acc:.4f} "
          f"- val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (best val_acc: {best_val_acc:.4f})")
            break

# restore_best_weights=True equivalent
if best_state is not None:
    model.load_state_dict(best_state)

Epoch   1/60 - loss: 2.0720 - acc: 0.3521 - val_loss: 1.2244 - val_acc: 0.6031
Epoch   2/60 - loss: 0.9539 - acc: 0.6802 - val_loss: 0.6422 - val_acc: 0.7828
Epoch   3/60 - loss: 0.6703 - acc: 0.7828 - val_loss: 0.4363 - val_acc: 0.8594
Epoch   4/60 - loss: 0.4996 - acc: 0.8416 - val_loss: 0.4102 - val_acc: 0.8891
Epoch   5/60 - loss: 0.4431 - acc: 0.8504 - val_loss: 0.2145 - val_acc: 0.9437
Epoch   6/60 - loss: 0.3528 - acc: 0.8791 - val_loss: 0.2617 - val_acc: 0.9125
Epoch   7/60 - loss: 0.3269 - acc: 0.8877 - val_loss: 0.2094 - val_acc: 0.9313
Epoch   8/60 - loss: 0.8233 - acc: 0.7472 - val_loss: 0.3618 - val_acc: 0.8797
Epoch   9/60 - loss: 0.4286 - acc: 0.8571 - val_loss: 0.2655 - val_acc: 0.9109
Epoch  10/60 - loss: 0.2962 - acc: 0.9084 - val_loss: 0.2810 - val_acc: 0.8938
Epoch  11/60 - loss: 0.3540 - acc: 0.9084 - val_loss: 0.3431 - val_acc: 0.8734
Epoch  12/60 - loss: 0.2916 - acc: 0.9081 - val_loss: 0.1569 - val_acc: 0.9469
Epoch  13/60 - loss: 0.1677 - acc: 0.9547 - val_loss

In [10]:
model.eval()
test_ds = TensorDataset(torch.from_numpy(X_test_s), torch.from_numpy(y_test).long())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

criterion_eval = nn.CrossEntropyLoss()
test_loss, test_correct, test_total = 0.0, 0, 0
all_preds = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = criterion_eval(out, yb)
        test_loss += loss.item() * xb.size(0)
        preds = out.argmax(1)
        test_correct += (preds == yb).sum().item()
        test_total += xb.size(0)
        all_preds.append(preds.cpu().numpy())

test_loss /= test_total
test_acc = test_correct / test_total
y_pred = np.concatenate(all_preds)

print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

target_names = [ACTIVITY_NAMES[inv_label_map[i]] for i in range(n_classes)]
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=target_names, digits=3))

Test accuracy: 0.9972
Test loss: 0.0236

Classification report:
                 precision    recall  f1-score   support

       Standing      1.000     1.000     1.000        96
        Sitting      0.990     1.000     0.995        96
          Lying      1.000     1.000     1.000        96
        Walking      1.000     1.000     1.000        96
Climbing stairs      1.000     0.989     0.995        94
    Waist bends      1.000     1.000     1.000        88
 Arms elevation      1.000     1.000     1.000        92
  Knees bending      1.000     1.000     1.000        91
        Cycling      1.000     1.000     1.000        96
        Jogging      0.990     0.990     0.990        96
        Running      0.990     1.000     0.995        96
Jump front&back      1.000     0.967     0.983        30

       accuracy                          0.997      1067
      macro avg      0.997     0.995     0.996      1067
   weighted avg      0.997     0.997     0.997      1067



In [11]:
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable params: {n_params}")
print("Per-layer param breakdown (needed later for CKKS ciphertext sizing):")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name:20s} {param.numel():>8d} params")

torch.save(model.state_dict(), "centralized_lstm_baseline.pt")
print("\nSaved model weights to centralized_lstm_baseline.pt")

Total trainable params: 36780
Per-layer param breakdown (needed later for CKKS ciphertext sizing):
  lstm1.weight_ih_l0       5888 params
  lstm1.weight_hh_l0      16384 params
  lstm1.bias_ih_l0          256 params
  lstm1.bias_hh_l0          256 params
  lstm2.weight_ih_l0       8192 params
  lstm2.weight_hh_l0       4096 params
  lstm2.bias_ih_l0          128 params
  lstm2.bias_hh_l0          128 params
  fc1.weight               1024 params
  fc1.bias                   32 params
  fc2.weight                384 params
  fc2.bias                   12 params

Saved model weights to centralized_lstm_baseline.pt


## 3. Federated baseline (FedAvg, no DP, no HE)

In [12]:
client_train, client_test = {}, {}
for subj, (Xw, yw) in client_data.items():
    yw_mapped = np.array([label_map[v] for v in yw])
    Xtr, Xte, ytr, yte = train_test_split(
        Xw, yw_mapped, test_size=0.2, stratify=yw_mapped, random_state=SEED
    )
    client_train[subj] = (Xtr, ytr)
    client_test[subj] = (Xte, yte)

# shared scaler fit on pooled training data across all clients (see note above)
fl_scaler = StandardScaler()
pooled_train_X = np.concatenate([Xtr for Xtr, _ in client_train.values()], axis=0)
fl_scaler.fit(pooled_train_X.reshape(-1, ch))

def fl_scale(X):
    shape = X.shape
    return fl_scaler.transform(X.reshape(-1, ch)).reshape(shape).astype(np.float32)

for subj in client_train:
    Xtr, ytr = client_train[subj]
    client_train[subj] = (fl_scale(Xtr), ytr)
    Xte, yte = client_test[subj]
    client_test[subj] = (fl_scale(Xte), yte)

X_test_global = np.concatenate([Xte for Xte, _ in client_test.values()], axis=0)
y_test_global = np.concatenate([yte for _, yte in client_test.values()], axis=0)

print(f"Clients: {len(client_train)}, n_classes: {n_classes}")
for subj, (Xtr, ytr) in client_train.items():
    print(f"  client {subj}: {len(ytr)} train windows, {len(client_test[subj][1])} test windows")

Clients: 10, n_classes: 12
  client 1: 438 train windows, 110 test windows
  client 2: 441 train windows, 111 test windows
  client 3: 438 train windows, 110 test windows
  client 4: 439 train windows, 110 test windows
  client 5: 423 train windows, 106 test windows
  client 6: 400 train windows, 101 test windows
  client 7: 421 train windows, 106 test windows
  client 8: 413 train windows, 104 test windows
  client 9: 428 train windows, 108 test windows
  client 10: 419 train windows, 105 test windows


In [13]:
import copy

def get_flat_state(model):
    return copy.deepcopy(model.state_dict())


def fedavg_aggregate(local_states, weights):
    """Weighted average of state_dicts, weighted by e.g. n_samples per client."""
    total = sum(weights)
    avg_state = copy.deepcopy(local_states[0])
    for key in avg_state:
        avg_state[key] = torch.zeros_like(avg_state[key], dtype=torch.float32)
        for state, w in zip(local_states, weights):
            avg_state[key] += state[key].float() * (w / total)
        avg_state[key] = avg_state[key].to(local_states[0][key].dtype)
    return avg_state


def local_train(model, X, y, epochs, batch_size, lr):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    ds = torch.utils.data.TensorDataset(torch.from_numpy(X), torch.from_numpy(y).long())
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            opt.step()
    return model


def fl_evaluate(model, X, y, batch_size=64):
    model.eval()
    ds = torch.utils.data.TensorDataset(torch.from_numpy(X), torch.from_numpy(y).long())
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False)
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    all_preds = []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * xb.size(0)
            preds = out.argmax(1)
            correct += (preds == yb).sum().item()
            total += xb.size(0)
            all_preds.append(preds.cpu().numpy())
    return total_loss / total, correct / total, np.concatenate(all_preds)

In [14]:
N_ROUNDS = 30
LOCAL_EPOCHS = 2
FL_BATCH_SIZE = 32
FL_LR = 1e-3
FL_PATIENCE = 6

global_model = build_lstm_model(n_channels=ch, n_classes=n_classes)
n_params_fl = sum(p.numel() for p in global_model.parameters() if p.requires_grad)
print(f"Global model params: {n_params_fl}")

fl_history = []
best_acc, best_state, no_improve = 0.0, None, 0

for rnd in range(1, N_ROUNDS + 1):
    local_states, local_weights = [], []

    for subj, (Xtr, ytr) in client_train.items():
        local_model = build_lstm_model(n_channels=ch, n_classes=n_classes)
        local_model.load_state_dict(get_flat_state(global_model))
        local_model = local_train(local_model, Xtr, ytr, LOCAL_EPOCHS, FL_BATCH_SIZE, FL_LR)
        local_states.append(get_flat_state(local_model))
        local_weights.append(len(ytr))

    new_global_state = fedavg_aggregate(local_states, local_weights)
    global_model.load_state_dict(new_global_state)

    val_loss, val_acc, _ = fl_evaluate(global_model, X_test_global, y_test_global)
    fl_history.append({"round": rnd, "test_loss": val_loss, "test_acc": val_acc})
    print(f"Round {rnd:3d}/{N_ROUNDS} - global test_loss: {val_loss:.4f} - global test_acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = copy.deepcopy(new_global_state)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= FL_PATIENCE:
            print(f"Early stopping at round {rnd} (best test_acc: {best_acc:.4f})")
            break

global_model.load_state_dict(best_state)

Global model params: 36780
Round   1/30 - global test_loss: 2.4242 - global test_acc: 0.2661
Round   2/30 - global test_loss: 2.1797 - global test_acc: 0.3249
Round   3/30 - global test_loss: 1.8672 - global test_acc: 0.3847
Round   4/30 - global test_loss: 1.5937 - global test_acc: 0.4631
Round   5/30 - global test_loss: 1.3619 - global test_acc: 0.5546
Round   6/30 - global test_loss: 1.1463 - global test_acc: 0.6480
Round   7/30 - global test_loss: 0.9791 - global test_acc: 0.7255
Round   8/30 - global test_loss: 0.9074 - global test_acc: 0.7796
Round   9/30 - global test_loss: 0.7891 - global test_acc: 0.7563
Round  10/30 - global test_loss: 0.6878 - global test_acc: 0.8077
Round  11/30 - global test_loss: 0.6431 - global test_acc: 0.8338
Round  12/30 - global test_loss: 0.5872 - global test_acc: 0.8441
Round  13/30 - global test_loss: 0.5493 - global test_acc: 0.8515
Round  14/30 - global test_loss: 0.5106 - global test_acc: 0.8693
Round  15/30 - global test_loss: 0.4533 - global 

<All keys matched successfully>

In [15]:
final_loss, final_acc, y_pred_fl = fl_evaluate(global_model, X_test_global, y_test_global)

print(f"=== FedAvg baseline (no DP/HE), {len(client_train)} clients ===")
print(f"Best global test accuracy: {final_acc:.4f}")
print(f"Best global test loss: {final_loss:.4f}")

target_names = [ACTIVITY_NAMES[inv_label_map[i]] for i in range(n_classes)]
print("\nClassification report (global model, pooled client test sets):")
print(classification_report(y_test_global, y_pred_fl, target_names=target_names, digits=3))

pd.DataFrame(fl_history).to_csv("fedavg_history.csv", index=False)
torch.save(global_model.state_dict(), "fedavg_global_model.pt")
print("\nSaved round history to fedavg_history.csv, weights to fedavg_global_model.pt")

=== FedAvg baseline (no DP/HE), 10 clients ===
Best global test accuracy: 0.9627
Best global test loss: 0.1743

Classification report (global model, pooled client test sets):
                 precision    recall  f1-score   support

       Standing      0.990     1.000     0.995        96
        Sitting      1.000     0.892     0.943        93
          Lying      1.000     1.000     1.000        97
        Walking      0.990     0.990     0.990       100
Climbing stairs      0.853     0.979     0.912        95
    Waist bends      0.952     0.920     0.936        87
 Arms elevation      1.000     1.000     1.000        89
  Knees bending      0.837     0.837     0.837        92
        Cycling      1.000     0.979     0.989        96
        Jogging      0.990     0.980     0.985        98
        Running      0.980     0.990     0.985        98
Jump front&back      1.000     1.000     1.000        30

       accuracy                          0.963      1071
      macro avg      0.96

## 4. Per-layer RDP accountant + budget calibration

In [16]:
import sys
!{sys.executable} -m pip install -q dp-accounting


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from dp_accounting import dp_event
from dp_accounting.rdp import rdp_privacy_accountant

RDP_ORDERS = (
    [1 + x / 10.0 for x in range(1, 100)]
    + list(range(11, 64))
    + [128, 256, 512, 1024]
)


def compute_epsilon_for_layer_sigmas(layer_sigmas, n_rounds, delta, sampling_probability=1.0):
    """
    Given per-layer noise multipliers (assumed constant across rounds for
    this accounting pass) and a number of rounds, return the achieved
    epsilon at the given delta. Each (round, layer) pair is one
    GaussianDpEvent(sigma_l), wrapped in a PoissonSampledDpEvent when
    sampling_probability < 1.0 to account for subsampling amplification;
    all events compose sequentially.
    """
    accountant = rdp_privacy_accountant.RdpAccountant(orders=RDP_ORDERS)
    events = []
    for sigma_l in layer_sigmas:
        gaussian_event = dp_event.GaussianDpEvent(noise_multiplier=sigma_l)
        if sampling_probability < 1.0:
            single_round_event = dp_event.PoissonSampledDpEvent(
                sampling_probability=sampling_probability, event=gaussian_event
            )
        else:
            single_round_event = gaussian_event
        events.append(dp_event.SelfComposedDpEvent(single_round_event, n_rounds))
    accountant.compose(dp_event.ComposedDpEvent(events))
    return accountant.get_epsilon(target_delta=delta)


def lan_rdp_layer_weights(grad_norm_ema, alpha):
    r"""sigma_i \propto (g_max / g_i) ^ alpha -- returns unnormalized per-layer weights w_l."""
    grad_norm_ema = np.asarray(grad_norm_ema, dtype=np.float64)
    g_max = grad_norm_ema.max()
    safe_norms = np.clip(grad_norm_ema, a_min=1e-8, a_max=None)
    return (g_max / safe_norms) ** alpha


def calibrate_uniform_sigma(target_epsilon, delta, n_layers, n_rounds,
                             sampling_probability=1.0,
                             sigma_lo=0.1, sigma_hi=50.0, tol=1e-3, max_iter=60):
    """Binary search a single sigma (same for every layer) hitting target_epsilon."""
    def eps_at(sigma):
        return compute_epsilon_for_layer_sigmas(
            [sigma] * n_layers, n_rounds, delta, sampling_probability=sampling_probability
        )

    eps_lo, eps_hi = eps_at(sigma_lo), eps_at(sigma_hi)
    if eps_lo < target_epsilon:
        raise ValueError(f"sigma_lo={sigma_lo} already gives eps={eps_lo:.4f} < target. Lower sigma_lo.")
    if eps_hi > target_epsilon:
        raise ValueError(f"sigma_hi={sigma_hi} still gives eps={eps_hi:.4f} > target. Raise sigma_hi.")

    lo, hi = sigma_lo, sigma_hi
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        eps_mid = eps_at(mid)
        if abs(eps_mid - target_epsilon) < tol:
            return mid, eps_mid
        if eps_mid > target_epsilon:
            lo = mid
        else:
            hi = mid
    return mid, eps_mid


def calibrate_lan_rdp_scale(target_epsilon, delta, layer_weights, n_rounds,
                             sampling_probability=1.0,
                             k_lo=0.01, k_hi=100.0, tol=1e-3, max_iter=60):
    """Binary search the global scale k such that sigma_l = k * layer_weights[l] hits target_epsilon."""
    layer_weights = np.asarray(layer_weights, dtype=np.float64)

    def eps_at(k):
        return compute_epsilon_for_layer_sigmas(
            (k * layer_weights).tolist(), n_rounds, delta, sampling_probability=sampling_probability
        )

    eps_lo, eps_hi = eps_at(k_lo), eps_at(k_hi)
    if eps_lo < target_epsilon:
        raise ValueError(f"k_lo={k_lo} already gives eps={eps_lo:.4f} < target. Lower k_lo.")
    if eps_hi > target_epsilon:
        raise ValueError(f"k_hi={k_hi} still gives eps={eps_hi:.4f} > target. Raise k_hi.")

    lo, hi = k_lo, k_hi
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        eps_mid = eps_at(mid)
        if abs(eps_mid - target_epsilon) < tol:
            return mid, eps_mid
        if eps_mid > target_epsilon:
            lo = mid
        else:
            hi = mid
    return mid, eps_mid

### Sanity check

In [24]:
N_LAYERS = 6          # placeholder count; swap for real per-tensor layout, see note below
DP_N_ROUNDS = 30
DELTA = 1e-5
TARGET_EPS = 4.0

print(f"=== Calibrating UNIFORM DP for target eps={TARGET_EPS}, delta={DELTA}, "
      f"{N_LAYERS} layers, {DP_N_ROUNDS} rounds ===")
sigma_u, eps_u = calibrate_uniform_sigma(TARGET_EPS, DELTA, N_LAYERS, DP_N_ROUNDS)
print(f"  sigma_uniform = {sigma_u:.4f}  ->  achieved eps = {eps_u:.4f}")

# placeholder gradient-norm EMA profile -- replace with the real EMA
# tracked during local training once the clip+noise step is wired in
fake_grad_norm_ema = np.array([2.5, 1.8, 1.2, 0.9, 0.4, 0.2])
ALPHA = 1.0

weights = lan_rdp_layer_weights(fake_grad_norm_ema, ALPHA)
print(f"\nLAN-RDP layer weights (alpha={ALPHA}): {np.round(weights, 3)}")

print(f"\n=== Calibrating LAN-RDP for the SAME target eps={TARGET_EPS} ===")
k, eps_lan = calibrate_lan_rdp_scale(TARGET_EPS, DELTA, weights, DP_N_ROUNDS)
sigmas_lan = k * weights
print(f"  global scale k = {k:.4f}  ->  achieved eps = {eps_lan:.4f}")
print(f"  per-layer sigmas: {np.round(sigmas_lan, 4)}")

print(f"\n=== Comparison ===")
print(f"  Uniform:  every layer sigma = {sigma_u:.4f}   (eps={eps_u:.4f})")
print(f"  LAN-RDP:  per-layer sigmas  = {np.round(sigmas_lan, 4)}   (eps={eps_lan:.4f})")
print(f"  sigma_lan / sigma_uniform per layer: {np.round(sigmas_lan / sigma_u, 3)}")

=== Calibrating UNIFORM DP for target eps=4.0, delta=1e-05, 6 layers, 30 rounds ===
  sigma_uniform = 15.5293  ->  achieved eps = 4.0003

LAN-RDP layer weights (alpha=1.0): [ 1.     1.389  2.083  2.778  6.25  12.5  ]

=== Calibrating LAN-RDP for the SAME target eps=4.0 ===
  global scale k = 8.7616  ->  achieved eps = 4.0009
  per-layer sigmas: [  8.7616  12.1688  18.2533  24.3377  54.7598 109.5196]

=== Comparison ===
  Uniform:  every layer sigma = 15.5293   (eps=4.0003)
  LAN-RDP:  per-layer sigmas  = [  8.7616  12.1688  18.2533  24.3377  54.7598 109.5196]   (eps=4.0009)
  sigma_lan / sigma_uniform per layer: [0.564 0.784 1.175 1.567 3.526 7.052]


## 5. Per-example DP-SGD (privacy unit = training window, not client)

In [25]:
import sys
!{sys.executable} -m pip install -q opacus


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from opacus.layers import DPLSTM
from opacus.grad_sample import GradSampleModule


class DPLSTMClassifier(nn.Module):
    """Same architecture as LSTMClassifier, but with DPLSTM in place of nn.LSTM
    so Opacus can compute per-sample gradients."""
    def __init__(self, n_channels, n_classes, hidden1=64, hidden2=32, dense=32, dropout=0.3):
        super().__init__()
        self.lstm1 = DPLSTM(n_channels, hidden1, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = DPLSTM(hidden1, hidden2, batch_first=True)
        self.drop2 = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden2, dense)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(dense, n_classes)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.drop1(out)
        out, _ = self.lstm2(out)
        out = out[:, -1, :]
        out = self.drop2(out)
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out


def build_dp_model(n_channels, n_classes):
    m = DPLSTMClassifier(n_channels, n_classes).to(device)
    return GradSampleModule(m)


def strip_prefix(name):
    return name[len("_module."):] if name.startswith("_module.") else name


def get_plain_state_dict(gm):
    return {strip_prefix(k): v.detach().clone() for k, v in gm.state_dict().items()}


def load_plain_state_dict(gm, state):
    gm._module.load_state_dict(state)

In [27]:
def dp_sgd_local_train(gm, X, y, epochs, batch_size, lr, clip_norms, sigma_dict, rng):
    """
    Manual per-example DP-SGD: per minibatch, compute per-example gradients,
    clip each example\'s per-layer gradient to clip_norms[layer], average over
    the batch, add Gaussian noise scaled by sigma_dict[layer]. Only parameters
    in clip_norms/sigma_dict are trained -- this excludes DPLSTM\'s inert
    internal parameter aliases (see markdown above).
    """
    active_names = set(clip_norms.keys())
    opt = torch.optim.Adam(
        [p for n, p in gm.named_parameters() if strip_prefix(n) in active_names], lr=lr
    )
    criterion = nn.CrossEntropyLoss()

    n = X.shape[0]
    for _ in range(epochs):
        perm = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            if len(idx) == 0:
                continue
            xb = torch.from_numpy(X[idx]).to(device)
            yb = torch.from_numpy(y[idx]).long().to(device)
            bsz = xb.shape[0]

            gm.zero_grad(set_to_none=True)
            out = gm(xb)
            loss = criterion(out, yb)
            loss.backward()

            avg_grads = {}
            for name, p in gm.named_parameters():
                plain_name = strip_prefix(name)
                if plain_name not in active_names:
                    del p.grad_sample
                    continue
                gs = p.grad_sample
                flat = gs.reshape(bsz, -1)
                norms = flat.norm(dim=1, keepdim=True)
                clip_c = clip_norms[plain_name]
                clip_coef = (clip_c / (norms + 1e-6)).clamp(max=1.0)
                clipped = flat * clip_coef
                summed = clipped.sum(dim=0)
                sigma = sigma_dict[plain_name]
                noise = torch.randn_like(summed) * sigma * clip_c
                noisy_sum = summed + noise
                avg_grads[plain_name] = (noisy_sum / bsz).reshape(p.shape)
                del p.grad_sample

            for name, p in gm.named_parameters():
                plain_name = strip_prefix(name)
                if plain_name in avg_grads:
                    p.grad = avg_grads[plain_name]

            opt.step()
    return gm


def evaluate_dp(gm, X, y, batch_size=64):
    gm.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds = []
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for start in range(0, len(y), batch_size):
            xb = torch.from_numpy(X[start:start + batch_size]).to(device)
            yb = torch.from_numpy(y[start:start + batch_size]).long().to(device)
            out = gm(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * xb.size(0)
            preds = out.argmax(1)
            correct += (preds == yb).sum().item()
            total += xb.size(0)
            all_preds.append(preds.cpu().numpy())
    gm.train()
    return total_loss / total, correct / total, np.concatenate(all_preds)


def average_state_dicts(states):
    n = len(states)
    avg = {}
    for k in states[0]:
        avg[k] = sum(s[k].float() for s in states) / n
        avg[k] = avg[k].to(states[0][k].dtype)
    return avg


def run_dp_sgd_fedavg(client_train, X_test, y_test, ch, n_classes, n_rounds,
                       local_epochs, batch_size, lr, clip_norms, sigma_dict,
                       condition_name, seed=SEED, patience=6):
    rng = np.random.default_rng(seed)
    global_gm = build_dp_model(ch, n_classes)
    global_state = get_plain_state_dict(global_gm)

    history = []
    best_acc, best_state, no_improve = 0.0, None, 0

    for rnd in range(1, n_rounds + 1):
        client_states = []
        for subj, (Xtr, ytr) in client_train.items():
            local_gm = build_dp_model(ch, n_classes)
            load_plain_state_dict(local_gm, global_state)
            local_gm = dp_sgd_local_train(local_gm, Xtr, ytr, local_epochs, batch_size, lr,
                                           clip_norms, sigma_dict, rng)
            client_states.append(get_plain_state_dict(local_gm))

        global_state = average_state_dicts(client_states)
        load_plain_state_dict(global_gm, global_state)

        val_loss, val_acc, _ = evaluate_dp(global_gm, X_test, y_test)
        history.append({"round": rnd, "test_loss": val_loss, "test_acc": val_acc})
        print(f"[{condition_name}] Round {rnd:3d}/{n_rounds} - test_loss: {val_loss:.4f} - test_acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc, best_state, no_improve = val_acc, copy.deepcopy(global_state), 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"[{condition_name}] Early stopping at round {rnd} (best test_acc: {best_acc:.4f})")
                break

    load_plain_state_dict(global_gm, best_state)
    return global_gm, history, best_acc

### Calibrate and run both conditions

In [28]:
N_ROUNDS_DPSGD = 8
LOCAL_EPOCHS_DPSGD = 1
BATCH_SIZE_DPSGD = 32
LR_DPSGD = 1e-3
DELTA_DPSGD = 1e-5
TARGET_EPS_DPSGD = 4.0
ALPHA_DPSGD = 1.0
TOTAL_CLIP_NORM = 1.0

avg_n_local = int(np.mean([len(ytr) for _, ytr in client_train.values()]))
steps_per_epoch = int(np.ceil(avg_n_local / BATCH_SIZE_DPSGD))
total_steps = N_ROUNDS_DPSGD * LOCAL_EPOCHS_DPSGD * steps_per_epoch
q_dpsgd = BATCH_SIZE_DPSGD / avg_n_local
print(f"avg client size: {avg_n_local}, steps/epoch: {steps_per_epoch}, "
      f"total_steps: {total_steps}, per-example sampling q: {q_dpsgd:.4f}")

print("\n=== Warm-up (no DP) for gradient-norm profile ===")
warmup_gm = build_dp_model(ch, n_classes)
warmup_global_state = get_plain_state_dict(warmup_gm)
layer_names_dp = list(warmup_global_state.keys())

grad_norm_accum = {k: 0.0 for k in layer_names_dp}
n_clients = len(client_train)
for subj, (Xtr, ytr) in client_train.items():
    local_gm = build_dp_model(ch, n_classes)
    load_plain_state_dict(local_gm, warmup_global_state)
    opt = torch.optim.Adam(local_gm.parameters(), lr=LR_DPSGD)
    criterion = nn.CrossEntropyLoss()
    xb = torch.from_numpy(Xtr[:BATCH_SIZE_DPSGD]).to(device)
    yb = torch.from_numpy(ytr[:BATCH_SIZE_DPSGD]).long().to(device)
    opt.zero_grad(set_to_none=True)
    out = local_gm(xb)
    loss = criterion(out, yb)
    loss.backward()
    for name, p in local_gm.named_parameters():
        plain_name = strip_prefix(name)
        g = p.grad_sample.reshape(p.grad_sample.shape[0], -1).norm(dim=1).mean().item()
        grad_norm_accum[plain_name] += g

grad_norm_ema_dp = np.array([grad_norm_accum[k] / n_clients for k in layer_names_dp])

# exclude DPLSTM\'s inert parameter aliases (zero warm-up gradient)
active_mask = grad_norm_ema_dp > 1e-6
dead_names = [k for k, m in zip(layer_names_dp, active_mask) if not m]
print(f"Excluding {len(dead_names)} inert DPLSTM parameter aliases: {dead_names}")
layer_names_dp = [k for k, m in zip(layer_names_dp, active_mask) if m]
grad_norm_ema_dp = grad_norm_ema_dp[active_mask]
n_layers_dp = len(layer_names_dp)

per_layer_clip = TOTAL_CLIP_NORM / np.sqrt(n_layers_dp)
clip_norms = {k: per_layer_clip for k in layer_names_dp}

print(f"\n=== Calibrating to eps={TARGET_EPS_DPSGD}, delta={DELTA_DPSGD}, "
      f"{n_layers_dp} layers, {total_steps} steps, q={q_dpsgd:.4f} ===")
sigma_u_dp, eps_u_dp = calibrate_uniform_sigma(
    TARGET_EPS_DPSGD, DELTA_DPSGD, n_layers_dp, total_steps, sampling_probability=q_dpsgd
)
print(f"Uniform:  sigma = {sigma_u_dp:.4f}  (achieved eps={eps_u_dp:.4f})")
uniform_sigma_dict_dp = {k: sigma_u_dp for k in layer_names_dp}

lan_weights_dp = lan_rdp_layer_weights(grad_norm_ema_dp, ALPHA_DPSGD)
k_scale_dp, eps_lan_dp = calibrate_lan_rdp_scale(
    TARGET_EPS_DPSGD, DELTA_DPSGD, lan_weights_dp, total_steps, sampling_probability=q_dpsgd
)
lan_sigma_dict_dp = {k: s for k, s in zip(layer_names_dp, k_scale_dp * lan_weights_dp)}
print(f"LAN-RDP:  k = {k_scale_dp:.4f}  (achieved eps={eps_lan_dp:.4f})")

avg client size: 426, steps/epoch: 14, total_steps: 112, per-example sampling q: 0.0751

=== Warm-up (no DP) for gradient-norm profile ===


C:\Users\rijik\AppData\Local\Temp\ipykernel_18228\62815688.py:34: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Excluding 8 inert DPLSTM parameter aliases: ['lstm1.l0.ih.weight', 'lstm1.l0.ih.bias', 'lstm1.l0.hh.weight', 'lstm1.l0.hh.bias', 'lstm2.l0.ih.weight', 'lstm2.l0.ih.bias', 'lstm2.l0.hh.weight', 'lstm2.l0.hh.bias']

=== Calibrating to eps=4.0, delta=1e-05, 12 layers, 112 steps, q=0.0751 ===
Uniform:  sigma = 3.3078  (achieved eps=4.0001)


08/19/2026 21:42:45:WARNING:_compute_log_a_frac failed to converge after 1000 iterations with q=0.075117, sigma=0.791172, alpha=1.100000. Excluding this order from the epsilon computation.
08/19/2026 21:42:45:WARNING:_compute_log_a_frac failed to converge after 1000 iterations with q=0.075117, sigma=0.791172, alpha=1.200000. Excluding this order from the epsilon computation.
08/19/2026 21:42:45:WARNING:_compute_log_a_frac failed to converge after 1000 iterations with q=0.075117, sigma=0.791172, alpha=1.300000. Excluding this order from the epsilon computation.
08/19/2026 21:42:45:WARNING:_compute_log_a_frac failed to converge after 1000 iterations with q=0.075117, sigma=0.791172, alpha=1.400000. Excluding this order from the epsilon computation.
08/19/2026 21:42:45:WARNING:_compute_log_a_frac failed to converge after 1000 iterations with q=0.075117, sigma=0.791172, alpha=1.500000. Excluding this order from the epsilon computation.


LAN-RDP:  k = 1.4514  (achieved eps=3.9995)


In [29]:
print(f"=== Running UNIFORM per-example DP-SGD FedAvg ({N_ROUNDS_DPSGD} rounds) ===")
_, hist_uniform_dp, acc_uniform_dp = run_dp_sgd_fedavg(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_DPSGD, LOCAL_EPOCHS_DPSGD, BATCH_SIZE_DPSGD, LR_DPSGD,
    clip_norms, uniform_sigma_dict_dp, "UNIFORM", seed=SEED
)

print(f"\n=== Running LAN-RDP per-example DP-SGD FedAvg ({N_ROUNDS_DPSGD} rounds) ===")
_, hist_lan_dp, acc_lan_dp = run_dp_sgd_fedavg(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_DPSGD, LOCAL_EPOCHS_DPSGD, BATCH_SIZE_DPSGD, LR_DPSGD,
    clip_norms, lan_sigma_dict_dp, "LAN-RDP", seed=SEED
)

print(f"\n=== RESULT at target eps={TARGET_EPS_DPSGD} (per-example DP) ===")
print(f"  Uniform best test accuracy: {acc_uniform_dp:.4f}  (achieved eps={eps_u_dp:.4f})")
print(f"  LAN-RDP  best test accuracy: {acc_lan_dp:.4f}  (achieved eps={eps_lan_dp:.4f})")
print(f"  Gap (LAN-RDP - Uniform): {acc_lan_dp - acc_uniform_dp:+.4f}")

pd.DataFrame(hist_uniform_dp).to_csv("dpsgd_uniform_history.csv", index=False)
pd.DataFrame(hist_lan_dp).to_csv("dpsgd_lan_rdp_history.csv", index=False)

=== Running UNIFORM per-example DP-SGD FedAvg (8 rounds) ===


C:\Users\rijik\AppData\Local\Temp\ipykernel_18228\2495966041.py:29: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


[UNIFORM] Round   1/8 - test_loss: 2.4856 - test_acc: 0.0896
[UNIFORM] Round   2/8 - test_loss: 2.4841 - test_acc: 0.0887
[UNIFORM] Round   3/8 - test_loss: 2.4825 - test_acc: 0.0887
[UNIFORM] Round   4/8 - test_loss: 2.4813 - test_acc: 0.0887
[UNIFORM] Round   5/8 - test_loss: 2.4798 - test_acc: 0.0896
[UNIFORM] Round   6/8 - test_loss: 2.4783 - test_acc: 0.0887
[UNIFORM] Round   7/8 - test_loss: 2.4770 - test_acc: 0.0887
[UNIFORM] Early stopping at round 7 (best test_acc: 0.0896)

=== Running LAN-RDP per-example DP-SGD FedAvg (8 rounds) ===
[LAN-RDP] Round   1/8 - test_loss: 2.4869 - test_acc: 0.0915
[LAN-RDP] Round   2/8 - test_loss: 2.4854 - test_acc: 0.0915
[LAN-RDP] Round   3/8 - test_loss: 2.4839 - test_acc: 0.0915
[LAN-RDP] Round   4/8 - test_loss: 2.4823 - test_acc: 0.0915
[LAN-RDP] Round   5/8 - test_loss: 2.4808 - test_acc: 0.0915
[LAN-RDP] Round   6/8 - test_loss: 2.4793 - test_acc: 0.0915
[LAN-RDP] Round   7/8 - test_loss: 2.4779 - test_acc: 0.0915
[LAN-RDP] Early stopping

## 6. Scaled-up comparison (15 rounds × 2 local epochs, 420 total DP-SGD steps)

In [ ]:
N_ROUNDS_SCALED = 15
LOCAL_EPOCHS_SCALED = 2

# same calibration approach as section 5, just with the larger total_steps
# implied by N_ROUNDS_SCALED * LOCAL_EPOCHS_SCALED * steps_per_epoch
total_steps_scaled = N_ROUNDS_SCALED * LOCAL_EPOCHS_SCALED * steps_per_epoch
q_scaled = BATCH_SIZE_DPSGD / avg_n_local
print(f"total_steps: {total_steps_scaled}, q: {q_scaled:.4f}")

sigma_u_scaled, eps_u_scaled = calibrate_uniform_sigma(
    TARGET_EPS_DPSGD, DELTA_DPSGD, n_layers_dp, total_steps_scaled, sampling_probability=q_scaled
)
uniform_sigma_dict_scaled = {k: sigma_u_scaled for k in layer_names_dp}

k_scale_scaled, eps_lan_scaled = calibrate_lan_rdp_scale(
    TARGET_EPS_DPSGD, DELTA_DPSGD, lan_weights_dp, total_steps_scaled, sampling_probability=q_scaled
)
lan_sigma_dict_scaled = {k: s for k, s in zip(layer_names_dp, k_scale_scaled * lan_weights_dp)}

print(f"Uniform:  sigma = {sigma_u_scaled:.4f}  (achieved eps={eps_u_scaled:.4f})")
print(f"LAN-RDP:  k = {k_scale_scaled:.4f}  (achieved eps={eps_lan_scaled:.4f})")

print(f"\n=== Running UNIFORM ({N_ROUNDS_SCALED} rounds x {LOCAL_EPOCHS_SCALED} local epochs) ===")
_, hist_uniform_scaled, acc_uniform_scaled = run_dp_sgd_fedavg(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, LR_DPSGD,
    clip_norms, uniform_sigma_dict_scaled, "UNIFORM", seed=SEED, patience=10
)

print(f"\n=== Running LAN-RDP ({N_ROUNDS_SCALED} rounds x {LOCAL_EPOCHS_SCALED} local epochs) ===")
_, hist_lan_scaled, acc_lan_scaled = run_dp_sgd_fedavg(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, LR_DPSGD,
    clip_norms, lan_sigma_dict_scaled, "LAN-RDP", seed=SEED, patience=10
)

pd.DataFrame(hist_uniform_scaled).to_csv("dpsgd_uniform_history_scaled.csv", index=False)
pd.DataFrame(hist_lan_scaled).to_csv("dpsgd_lan_rdp_history_scaled.csv", index=False)
print("\nSaved scaled histories.")

### Results from the scaled run

Both conditions completed all 15 rounds (`run_dp_sgd_fedavg`'s `best_acc` tracks the peak across every round, which is what's reported below.

| round | Uniform test_acc | LAN-RDP test_acc |
|---|---|---|
| 1 | 8.68% | 8.68% |
| 5 | 8.68% | 8.68% |
| 8 | 8.78% | 9.15% |
| 9 | 9.06% | 9.52% |
| 10 | 9.06% | 10.08% |
| 11 | 9.34% | **10.27%** ← LAN-RDP peak |
| 12 | 9.43% | 9.71% |
| 13 | 9.71% | 9.24% |
| 14 | 9.90% | 8.22% |
| 15 | **10.08%** ← Uniform peak | 6.63% |



## 7. Stability fix attempt: capping the LAN-RDP allocation ratio

In [ ]:
MAX_WEIGHT = 4.0
lan_weights_capped = lan_rdp_layer_weights(grad_norm_ema_dp, ALPHA_DPSGD, max_weight=MAX_WEIGHT)
k_scale_capped, eps_lan_capped = calibrate_lan_rdp_scale(
    TARGET_EPS_DPSGD, DELTA_DPSGD, lan_weights_capped, total_steps, sampling_probability=q_dpsgd
)
lan_capped_sigma_dict = {k: s for k, s in zip(layer_names_dp, k_scale_capped * lan_weights_capped)}
print(f"LAN-RDP (capped, max_weight={MAX_WEIGHT}): k={k_scale_capped:.4f} (achieved eps={eps_lan_capped:.4f})")

print(f"\n=== Running LAN-RDP (capped) ({N_ROUNDS_SCALED} rounds x {LOCAL_EPOCHS_SCALED} local epochs) ===")
_, hist_lan_capped, acc_lan_capped = run_dp_sgd_fedavg(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, LR_DPSGD,
    clip_norms, lan_capped_sigma_dict, "LAN-RDP-CAPPED", seed=SEED, patience=10
)
pd.DataFrame(hist_lan_capped).to_csv("dpsgd_lan_rdp_capped_history_scaled.csv", index=False)

## 8. Testing the Adam-vs-SGD hypothesis

In [ ]:
SGD_LR = 0.05
SGD_MOMENTUM = 0.9

print(f"=== Running UNIFORM with SGD ({N_ROUNDS_SCALED} rounds x {LOCAL_EPOCHS_SCALED} local epochs) ===")
_, hist_uniform_sgd, acc_uniform_sgd = run_dp_sgd_fedavg(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, SGD_LR,
    clip_norms, uniform_sigma_dict_scaled, "UNIFORM-SGD", seed=SEED, patience=10
)
pd.DataFrame(hist_uniform_sgd).to_csv("dpsgd_uniform_sgd_history_scaled.csv", index=False)

### Result: SGD is dramatically more stable for Uniform DP

| round | Adam | SGD |
|---|---|---|
| 1 | 8.68% | 8.68% |
| 2 | 8.68% | **14.75%** |
| 3 | 8.68% | **26.98%** |
| 4 | 8.68% | **32.31%** |
| 5 | 8.68% | **38.10%** |
| 6 | 8.68% | **39.22%** ← peak |
| 10 | 9.06% | 35.48% |
| 15 | **10.08%** | 26.89% |



In [ ]:
print(f"=== Running LAN-RDP with SGD ({N_ROUNDS_SCALED} rounds x {LOCAL_EPOCHS_SCALED} local epochs) ===")
_, hist_lan_sgd, acc_lan_sgd = run_dp_sgd_fedavg(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, SGD_LR,
    clip_norms, lan_sigma_dict_scaled, "LAN-RDP-SGD", seed=SEED, patience=10
)
pd.DataFrame(hist_lan_sgd).to_csv("dpsgd_lan_rdp_sgd_history_scaled.csv", index=False)

### Result: LAN-RDP diverges to NaN under SGD

| round | test_loss | test_acc |
|---|---|---|
| 1 | 2.4726 | 7.94% |
| 2 | 2.4698 | 8.31% |
| 3 | 2.4641 | 10.55% |
| 4 | 2.4752 | **10.74%** ← peak |
| 5 | 2.5047 | 10.46% |
| 6 | **NaN** | 8.96% |
| 7 | **NaN** | 8.96% (frozen — run stopped here, no recovery possible) |


## 9. CKKS + ASCON-128 hybrid layer

In [ ]:
import sys
!{sys.executable} -m pip install -q tenseal ascon

In [ ]:
import os
import time
import tenseal as ts
import ascon

MAX_SLOTS = 4096  # poly_modulus_degree=8192 -> 4096 CKKS slots per ciphertext


def build_ckks_context(poly_modulus_degree=8192, coeff_mod_bit_sizes=(60, 40, 40, 60), global_scale=2**40):
    """Aggregator-side: generates the CKKS keypair. Edge devices get a public
    (secret-key-stripped) copy for encryption only."""
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=poly_modulus_degree,
        coeff_mod_bit_sizes=list(coeff_mod_bit_sizes),
    )
    context.global_scale = global_scale
    context.generate_galois_keys()
    return context


def encrypt_layer(context, flat_array, max_slots=MAX_SLOTS):
    """Encrypt a flat numpy array as one or more CKKS ciphertext chunks
    (a layer's parameter count can exceed max_slots)."""
    chunks = []
    n = len(flat_array)
    for start in range(0, n, max_slots):
        chunk = flat_array[start:start + max_slots]
        chunks.append(ts.ckks_vector(context, chunk.tolist()))
    return chunks


def decrypt_layer(chunks, original_length):
    parts = [np.array(c.decrypt()) for c in chunks]
    return np.concatenate(parts)[:original_length]


def ascon_wrap(key, plaintext_bytes, associated_data=b""):
    """Edge-device transport step: authenticate + encrypt a serialized
    ciphertext chunk before upload."""
    nonce = os.urandom(16)
    wrapped = ascon.encrypt(key, nonce, associated_data, plaintext_bytes, variant="Ascon-128")
    return nonce, wrapped


def ascon_unwrap(key, nonce, wrapped_bytes, associated_data=b""):
    """Aggregator-side: verify + strip. Raises on tamper/auth failure."""
    return ascon.decrypt(key, nonce, associated_data, wrapped_bytes, variant="Ascon-128")


def client_encrypt_and_wrap(context, ascon_key, layer_deltas, device_id):
    """Full edge-device pipeline for one client's round update."""
    payload = {}
    byte_log = []
    for layer_name, tensor in layer_deltas.items():
        flat = tensor.detach().cpu().numpy().astype(np.float64).flatten()
        chunks = encrypt_layer(context, flat)
        wrapped_chunks = []
        for chunk in chunks:
            serialized = chunk.serialize()
            nonce, wrapped = ascon_wrap(ascon_key, serialized, associated_data=device_id.encode())
            wrapped_chunks.append((nonce, wrapped, len(flat)))
            byte_log.append({
                "device_id": device_id, "layer": layer_name,
                "raw_float32_bytes": flat.nbytes,
                "ckks_serialized_bytes": len(serialized),
                "ascon_wrapped_bytes": len(wrapped),
                "ascon_overhead_bytes": len(wrapped) - len(serialized),
            })
        payload[layer_name] = wrapped_chunks
    return payload, byte_log


def server_unwrap_and_aggregate(context, ascon_key, client_payloads, layer_shapes, n_clients):
    """ASCON-verify+strip each client's chunks, homomorphically SUM across
    clients, scalar-multiply by 1/n_clients, decrypt ONCE per layer."""
    aggregated = {}
    for layer_name in layer_shapes:
        summed_chunks = None
        for client_payload in client_payloads:
            device_id = client_payload["_device_id"]
            client_chunks = []
            for nonce, wrapped, chunk_len in client_payload[layer_name]:
                serialized = ascon_unwrap(ascon_key, nonce, wrapped, associated_data=device_id.encode())
                client_chunks.append(ts.ckks_vector_from(context, serialized))
            if summed_chunks is None:
                summed_chunks = client_chunks
            else:
                summed_chunks = [s + c for s, c in zip(summed_chunks, client_chunks)]
        avg_chunks = [c * (1.0 / n_clients) for c in summed_chunks]
        flat_len = int(np.prod(layer_shapes[layer_name]))
        aggregated[layer_name] = decrypt_layer(avg_chunks, flat_len).reshape(layer_shapes[layer_name])
    return aggregated

### Correctness + communication-cost demo


In [ ]:
layer_shapes = {k: tuple(v.shape) for k, v in get_plain_state_dict(build_model(ch, n_classes)).items()
                if k in clip_norms}  # active (non-inert) layers only, from section 5
n_clients_demo = len(client_train)  # 10, matching the real FL setup

print("Building CKKS context (aggregator-side keypair)...")
t0 = time.time()
context_full = build_ckks_context()
print(f"  context built in {time.time()-t0:.2f}s")

context_public = context_full.copy()
context_public.make_context_public()  # what edge devices actually hold

ascon_key = os.urandom(16)  # shared/provisioned device key (per-device in a real deployment)

# stand-in clipped+noised deltas, one set per client, matching real layer shapes
rng_demo = np.random.default_rng(0)
client_deltas_demo = []
for i in range(n_clients_demo):
    deltas = {name: torch.from_numpy((rng_demo.standard_normal(shape) * 0.1).astype(np.float32))
              for name, shape in layer_shapes.items()}
    client_deltas_demo.append(deltas)

plaintext_avg = {name: np.mean([d[name].numpy() for d in client_deltas_demo], axis=0) for name in layer_shapes}

In [ ]:
print(f"Encrypting + ASCON-wrapping {n_clients_demo} clients\' updates...")
t0 = time.time()
all_payloads, all_byte_logs = [], []
for i, deltas in enumerate(client_deltas_demo):
    device_id = f"device_{i}"
    payload, byte_log = client_encrypt_and_wrap(context_public, ascon_key, deltas, device_id)
    payload["_device_id"] = device_id
    all_payloads.append(payload)
    all_byte_logs.extend(byte_log)
encrypt_time = time.time() - t0
print(f"  encrypted {n_clients_demo} clients in {encrypt_time:.2f}s ({encrypt_time/n_clients_demo:.2f}s/client)")

print("\nAggregator: unwrap + homomorphic FedAvg + single decrypt...")
t0 = time.time()
decrypted_avg = server_unwrap_and_aggregate(context_full, ascon_key, all_payloads, layer_shapes, n_clients_demo)
aggregate_time = time.time() - t0
print(f"  aggregated in {aggregate_time:.2f}s")

print("\n=== Correctness check (decrypted HE aggregate vs plaintext FedAvg average) ===")
for name in layer_shapes:
    err = np.abs(decrypted_avg[name] - plaintext_avg[name]).max()
    print(f"  {name:20s} max abs error: {err:.2e}")

print("\n=== Communication cost (per round, all clients, all layers) ===")
total_raw = sum(b["raw_float32_bytes"] for b in all_byte_logs)
total_ckks = sum(b["ckks_serialized_bytes"] for b in all_byte_logs)
total_ascon_overhead = sum(b["ascon_overhead_bytes"] for b in all_byte_logs)
total_wrapped = sum(b["ascon_wrapped_bytes"] for b in all_byte_logs)
print(f"  Raw float32 payload (plain FedAvg, no HE):  {total_raw:>10,} bytes")
print(f"  CKKS serialized payload:                    {total_ckks:>10,} bytes  ({total_ckks/total_raw:.1f}x raw)")
print(f"  + ASCON overhead:                            {total_ascon_overhead:>10,} bytes")
print(f"  Total uploaded (CKKS + ASCON):               {total_wrapped:>10,} bytes  ({total_wrapped/total_raw:.1f}x raw)")

pd.DataFrame(all_byte_logs).to_csv("ckks_ascon_byte_log.csv", index=False)

## 10. Top-k sparsification combined with CKKS + ASCON

In [ ]:
def compute_topk_masks(warmup_delta, layer_names, sparsity_ratios):
    """sparsity_ratios: fraction of values to KEEP per layer. Returns
    {ratio: {layer_name: sorted_index_array}}."""
    masks = {}
    for ratio in sparsity_ratios:
        layer_masks = {}
        for name in layer_names:
            flat = warmup_delta[name]
            k = max(1, int(len(flat) * ratio))
            top_idx = np.argsort(np.abs(flat))[-k:]
            layer_masks[name] = np.sort(top_idx)
        masks[ratio] = layer_masks
    return masks


def client_encrypt_sparse_and_wrap(context, ascon_key, layer_deltas, layer_masks, device_id):
    """Only the masked (surviving) values are extracted and encrypted --
    fewer values means fewer/smaller CKKS chunks, which is the actual
    payload saving. Byte counts logged once per layer, not per chunk."""
    payload = {}
    byte_log = []
    for layer_name, tensor in layer_deltas.items():
        flat_full = tensor.detach().cpu().numpy().astype(np.float64).flatten()
        mask_idx = layer_masks[layer_name]
        sparse_values = flat_full[mask_idx]

        chunks = encrypt_layer(context, sparse_values)
        wrapped_chunks = []
        layer_ckks_bytes = 0
        for chunk in chunks:
            serialized = chunk.serialize()
            nonce, wrapped = ascon_wrap(ascon_key, serialized, associated_data=device_id.encode())
            wrapped_chunks.append((nonce, wrapped, len(sparse_values)))
            layer_ckks_bytes += len(wrapped)
        payload[layer_name] = wrapped_chunks
        byte_log.append({
            "device_id": device_id, "layer": layer_name,
            "n_kept": len(sparse_values), "n_total": len(flat_full),
            "raw_dense_bytes": flat_full.nbytes,
            "raw_sparse_values_only_bytes": sparse_values.nbytes,
            "ckks_serialized_and_wrapped_bytes": layer_ckks_bytes,
        })
    return payload, byte_log


def server_unwrap_and_aggregate_sparse(context, ascon_key, client_payloads, layer_shapes, layer_masks, n_clients):
    """Aggregate the sparse ciphertexts, decrypt once per layer, scatter
    back into full shape (unmasked positions = 0, i.e. frozen this round)."""
    aggregated = {}
    for layer_name, shape in layer_shapes.items():
        summed_chunks = None
        for client_payload in client_payloads:
            device_id = client_payload["_device_id"]
            client_chunks = []
            for nonce, wrapped, chunk_len in client_payload[layer_name]:
                serialized = ascon_unwrap(ascon_key, nonce, wrapped, associated_data=device_id.encode())
                client_chunks.append(ts.ckks_vector_from(context, serialized))
            if summed_chunks is None:
                summed_chunks = client_chunks
            else:
                summed_chunks = [s + c for s, c in zip(summed_chunks, client_chunks)]
        avg_chunks = [c * (1.0 / n_clients) for c in summed_chunks]
        n_kept = len(layer_masks[layer_name])
        decrypted_sparse = decrypt_layer(avg_chunks, n_kept)
        full = np.zeros(int(np.prod(shape)), dtype=np.float64)
        full[layer_masks[layer_name]] = decrypted_sparse
        aggregated[layer_name] = full.reshape(shape)
    return aggregated

### Build the mask from the warm-up gradient profile, then sweep sparsity levels

In [ ]:
# reuse the warm-up mechanism from section 5, but keep the actual per-parameter
# delta (not just its norm) so top-k can select individual values, not whole layers
warmup_gm2 = build_model(ch, n_classes)
warmup_state2 = get_plain_state_dict(warmup_gm2)
delta_accum = {k: np.zeros(v.numel()) for k, v in warmup_state2.items()}
for subj, (Xtr, ytr) in client_train.items():
    local_gm = build_model(ch, n_classes)
    load_plain_state_dict(local_gm, warmup_state2)
    opt = torch.optim.Adam(local_gm.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    xb = torch.from_numpy(Xtr[:32]).to(device)
    yb = torch.from_numpy(ytr[:32]).long().to(device)
    opt.zero_grad(set_to_none=True)
    out = local_gm(xb)
    loss = criterion(out, yb)
    loss.backward()
    for name, p in local_gm.named_parameters():
        plain_name = strip_prefix(name)
        g = p.grad_sample.mean(dim=0).detach().cpu().numpy().flatten()
        delta_accum[plain_name] += g / len(client_train)

warmup_delta = {k: delta_accum[k] for k in layer_names_dp}  # active layers only, from section 5
SPARSITY_RATIOS = [1.0, 0.5, 0.2, 0.1]
masks = compute_topk_masks(warmup_delta, layer_names_dp, SPARSITY_RATIOS)

layer_shapes_10 = {k: tuple(warmup_state2[k].shape) for k in layer_names_dp}
n_clients_sweep = 3  # scaled down from 10 purely for sandbox runtime; per-client bytes are unaffected
rng10 = np.random.default_rng(1)
client_deltas_demo = [
    {name: torch.from_numpy((rng10.standard_normal(shape) * 0.1).astype(np.float32))
     for name, shape in layer_shapes_10.items()}
    for _ in range(n_clients_sweep)
]

In [ ]:
print(f"{\'sparsity\':>10} {\'n_kept\':>10} {\'raw_dense_KB\':>14} {\'raw_sparse_KB\':>15} "
      f"{\'ckks_KB\':>12} {\'ckks_vs_raw_dense\':>18}")
sparsify_results = []
for ratio in SPARSITY_RATIOS:
    layer_masks = masks[ratio]
    all_payloads, all_byte_logs = [], []
    for i, deltas in enumerate(client_deltas_demo):
        device_id = f"device_{i}"
        payload, byte_log = client_encrypt_sparse_and_wrap(context_public, ascon_key, deltas, layer_masks, device_id)
        payload["_device_id"] = device_id
        all_payloads.append(payload)
        all_byte_logs.extend(byte_log)

    total_raw_dense = sum(b["raw_dense_bytes"] for b in all_byte_logs) / n_clients_sweep
    total_raw_sparse = sum(b["raw_sparse_values_only_bytes"] for b in all_byte_logs) / n_clients_sweep
    total_ckks = sum(b["ckks_serialized_and_wrapped_bytes"] for b in all_byte_logs) / n_clients_sweep
    n_kept_total = sum(b["n_kept"] for b in all_byte_logs) // n_clients_sweep // len(layer_shapes_10) * len(layer_shapes_10)

    print(f"{ratio:>10.0%} {n_kept_total:>10,} {total_raw_dense/1024:>14.1f} {total_raw_sparse/1024:>15.1f} "
          f"{total_ckks/1024:>12.1f} {total_ckks/total_raw_dense:>17.2f}x")
    sparsify_results.append({
        "sparsity_keep_ratio": ratio, "n_kept": n_kept_total,
        "raw_dense_bytes": total_raw_dense, "raw_sparse_bytes": total_raw_sparse,
        "ckks_bytes": total_ckks, "ckks_vs_raw_dense_ratio": total_ckks / total_raw_dense,
    })

pd.DataFrame(sparsify_results).to_csv("sparsify_ckks_communication_cost.csv", index=False)

### Results

| sparsity kept | n_kept | raw dense | raw sparse (values only) | CKKS+ASCON | ratio vs. raw dense |
|---|---|---|---|---|---|
| 100% | 36,780 | 287.3 KB | 287.3 KB | 5,500.3 KB | 19.14x |
| 50% | 18,390 | 287.3 KB | 143.7 KB | 4,208.0 KB | 14.64x |
| 20% | 7,350 | 287.3 KB | 57.4 KB | 3,883.2 KB | 13.51x |
| 10% | 3,672 | 287.3 KB | 28.7 KB | 3,883.4 KB | 13.51x |


## 11. Validating sparsification's accuracy cost

In [ ]:
def topk_sparsify_delta(delta_dict, ratio):
    """Dynamic, per-client, per-round top-k sparsification of a client's
    round delta (local_state - global_state), applied AFTER clip+noise.
    No HE alignment constraint here (plaintext experiment) -- each client
    keeps its own top-k positions independently."""
    if ratio >= 1.0:
        return delta_dict
    sparse = {}
    for name, tensor in delta_dict.items():
        flat = tensor.flatten()
        k = max(1, int(flat.numel() * ratio))
        if k >= flat.numel():
            sparse[name] = tensor
            continue
        topk_idx = torch.topk(flat.abs(), k).indices
        mask = torch.zeros_like(flat)
        mask[topk_idx] = 1.0
        sparse[name] = (flat * mask).reshape(tensor.shape)
    return sparse


def run_dp_sgd_fedavg_sparse(client_train, X_test, y_test, ch, n_classes, n_rounds,
                              local_epochs, batch_size, lr, clip_norms, sigma_dict,
                              sparsify_ratio, condition_name, seed=SEED, patience=10):
    rng = np.random.default_rng(seed)
    global_gm = build_dp_model(ch, n_classes)
    global_state = get_plain_state_dict(global_gm)

    history = []
    best_acc, best_state, no_improve = 0.0, None, 0

    for rnd in range(1, n_rounds + 1):
        deltas = []
        for subj, (Xtr, ytr) in client_train.items():
            local_gm = build_dp_model(ch, n_classes)
            load_plain_state_dict(local_gm, global_state)
            local_gm = dp_sgd_local_train(local_gm, Xtr, ytr, local_epochs, batch_size, lr,
                                           clip_norms, sigma_dict, rng, optimizer_type="sgd", momentum=0.9)
            local_state = get_plain_state_dict(local_gm)
            delta = {k: (local_state[k].float() - global_state[k].float()) for k in global_state}
            delta = topk_sparsify_delta(delta, sparsify_ratio)
            deltas.append(delta)

        n = len(deltas)
        for k in global_state:
            avg_delta = sum(d[k].float() for d in deltas) / n
            global_state[k] = (global_state[k].float() + avg_delta).to(global_state[k].dtype)
        load_plain_state_dict(global_gm, global_state)

        val_loss, val_acc, _ = evaluate_dp(global_gm, X_test, y_test)
        history.append({"round": rnd, "test_loss": val_loss, "test_acc": val_acc})
        print(f"[{condition_name}] Round {rnd:3d}/{n_rounds} - test_loss: {val_loss:.4f} - test_acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc, best_state, no_improve = val_acc, copy.deepcopy(global_state), 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"[{condition_name}] Early stopping at round {rnd} (best test_acc: {best_acc:.4f})")
                break

    load_plain_state_dict(global_gm, best_state)
    return global_gm, history, best_acc

In [ ]:
print("=== Running 50% sparsified (SGD) ===")
_, hist_50, acc_50 = run_dp_sgd_fedavg_sparse(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, SGD_LR,
    clip_norms, uniform_sigma_dict_scaled, 0.5, "SPARSE-50", seed=SEED
)

print("\n=== Running 20% sparsified (SGD) ===")
_, hist_20, acc_20 = run_dp_sgd_fedavg_sparse(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, SGD_LR,
    clip_norms, uniform_sigma_dict_scaled, 0.2, "SPARSE-20", seed=SEED
)

pd.DataFrame(hist_50).to_csv("dpsgd_uniform_sgd_sparse50_history.csv", index=False)
pd.DataFrame(hist_20).to_csv("dpsgd_uniform_sgd_sparse20_history.csv", index=False)

### Results: sparsification barely hurts peak accuracy, and dramatically improves late-training stability

| round | 100% (no sparsify) | 50% keep | 20% keep |
|---|---|---|---|
| 2 | 14.75% | 17.18% | 9.34% |
| 4 | 32.31% | 31.28% | 33.05% |
| 6 | **39.22%** ← 100% peak | 39.68% | 36.04% |
| 8 | 36.51% | 39.87% | 38.19% |
| 9 | 35.85% | **40.24%** ← 50% peak | 37.63% |
| 12 | 33.15% | 33.33% | 36.88% |
| 15 | 26.89% | 27.92% | **35.20%** |

| condition | peak accuracy | final (round 15) accuracy | peak-to-final decline |
|---|---|---|---|
| 100% (no sparsify) | 39.22% | 26.89% | 12.33pt (31.4% relative) |
| 50% keep | **40.24%** (best peak) | 27.92% | 12.32pt (30.6% relative) |
| 20% keep | 38.19% | **35.20%** (best final) | **2.99pt (7.8% relative)** |


## 12. LAN-RDP + sparsification: does the stability benefit rescue it?


In [ ]:
print("=== Running LAN-RDP (capped) + SGD + 20% sparsified ===")
_, hist_lan_capped_sparse20, acc_lan_capped_sparse20 = run_dp_sgd_fedavg_sparse(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, SGD_LR,
    clip_norms, lan_capped_sigma_dict, 0.2, "LAN-RDP-CAPPED-SGD-SPARSE20", seed=SEED
)
pd.DataFrame(hist_lan_capped_sparse20).to_csv("dpsgd_lan_capped_sgd_sparse20_history.csv", index=False)

### Result: stable, but not competitive

| round | Uniform+SGD+20% (section 11) | LAN-RDP(capped)+SGD+20% |
|---|---|---|
| 3 | 27.92% | 22.13% |
| 4 | 33.05% | **29.69% ← peak** |
| 5 | 35.20% | 28.57% |
| 8 | 38.19% | 24.00% |
| 10 | 37.91% | 21.76% |
| 14 (early stop) | (still running) | 19.51% |

## 13. Poisoning / Byzantine-robustness evaluation

In [ ]:
def poison_labels(y, n_classes):
    """Cyclic label-shift poisoning: every label relabeled to (label+1) mod n_classes."""
    return (y + 1) % n_classes


def average_state_dicts_plain(states):
    n = len(states)
    avg = {}
    for k in states[0]:
        avg[k] = torch.zeros_like(states[0][k], dtype=torch.float32)
        for state in states:
            avg[k] += state[k].float() / n
        avg[k] = avg[k].to(states[0][k].dtype)
    return avg


def trimmed_mean_state_dicts(states, trim_fraction):
    """Coordinate-wise trimmed mean -- the Byzantine-robust rule named in
    the lit review (Krum, Trimmed Mean, FLTrust -- Kalapaaking et al. 2023).
    For each parameter, sort client values and discard the top/bottom
    trim_fraction before averaging what remains."""
    n = len(states)
    n_trim = int(np.floor(n * trim_fraction))
    avg = {}
    for k in states[0]:
        stacked = torch.stack([s[k].float() for s in states], dim=0)
        sorted_vals, _ = torch.sort(stacked, dim=0)
        if n_trim > 0 and (n - 2 * n_trim) > 0:
            trimmed = sorted_vals[n_trim: n - n_trim]
        else:
            trimmed = sorted_vals
        avg[k] = trimmed.mean(dim=0).to(states[0][k].dtype)
    return avg


def run_fedavg_with_attack(client_train, X_test, y_test, ch, n_classes,
                            malicious_ids, aggregation, n_rounds, condition_name):
    global_model = build_lstm_model(ch, n_classes)
    global_state = get_flat_state(global_model)
    history = []

    for rnd in range(1, n_rounds + 1):
        client_states = []
        for subj, (Xtr, ytr) in client_train.items():
            ytr_use = poison_labels(ytr, n_classes) if subj in malicious_ids else ytr
            local_model = build_lstm_model(ch, n_classes)
            local_model.load_state_dict(global_state)
            local_model = local_train(local_model, Xtr, ytr_use, LOCAL_EPOCHS, BATCH_SIZE, LR)
            client_states.append(get_flat_state(local_model))

        if aggregation == "mean":
            global_state = average_state_dicts_plain(client_states)
        else:
            global_state = trimmed_mean_state_dicts(client_states, TRIM_FRACTION)

        global_model.load_state_dict(global_state)
        val_loss, val_acc, _ = evaluate(global_model, X_test, y_test)
        history.append({"round": rnd, "test_loss": val_loss, "test_acc": val_acc})
        print(f"[{condition_name}] Round {rnd:3d}/{n_rounds} - test_loss: {val_loss:.4f} - test_acc: {val_acc:.4f}")

    return global_model, history

In [ ]:
N_MALICIOUS = 3       # out of 10 clients (30%)
TRIM_FRACTION = 0.3   # trim 30% from each end -> keep middle 4 of 10
N_ROUNDS_POISON = 15

subject_ids = sorted(client_train.keys(), key=lambda x: int(x))
rng_poison = np.random.default_rng(SEED)
malicious_ids = set(rng_poison.choice(subject_ids, size=N_MALICIOUS, replace=False))
print(f"Malicious clients ({N_MALICIOUS}/{len(subject_ids)}): {sorted(malicious_ids)}")

print("\n=== Condition A: FedAvg (plain mean) WITH poisoning attack ===")
_, hist_attacked_mean = run_fedavg_with_attack(
    client_train, X_test_global, y_test_global, ch, n_classes,
    malicious_ids, "mean", N_ROUNDS_POISON, "ATTACKED-MEAN"
)

print("\n=== Condition B: Trimmed Mean WITH poisoning attack (same malicious clients) ===")
_, hist_attacked_trimmed = run_fedavg_with_attack(
    client_train, X_test_global, y_test_global, ch, n_classes,
    malicious_ids, "trimmed_mean", N_ROUNDS_POISON, "ATTACKED-TRIMMED"
)

pd.DataFrame(hist_attacked_mean).to_csv("poison_attacked_mean_history.csv", index=False)
pd.DataFrame(hist_attacked_trimmed).to_csv("poison_attacked_trimmed_history.csv", index=False)

### Results: Trimmed Mean beat plain FedAvg at every single round

| round | plain mean (vulnerable) | Trimmed Mean (defended) |
|---|---|---|
| 1 | 8.22% | 20.45% |
| 3 | 45.75% | 53.13% |
| 5 | 58.64% | 66.48% |
| 8 | 69.19% | 73.58% |
| 10 | 72.92% | 76.94% |
| 12 | 75.35% | 81.33% |
| 15 | **78.15%** (final) | **84.41%** (final) |

| condition | best accuracy | final (round 15) accuracy |
|---|---|---|
| Clean baseline, no attack (section 3, 30 rounds) | — | 97.5% |
| Attacked, plain mean aggregation | 78.80% | 78.15% |
| Attacked, Trimmed Mean | **84.41%** | **84.41%** |

## 14. Weighted Clip-Norm Split: A New Hypothesis, and LAN-RDP's First Win


In [ ]:
def weighted_clip_split(layer_names, grad_norm_ema, total_clip):
    """C_l proportional to the layer's own natural gradient norm, normalized
    so sum(C_l^2) = total_clip^2 exactly (preserves overall sensitivity)."""
    g = np.asarray(grad_norm_ema, dtype=np.float64)
    norm_g = np.sqrt((g ** 2).sum())
    c_l = total_clip * (g / norm_g)
    return {k: float(c) for k, c in zip(layer_names, c_l)}


def uniform_clip_split(layer_names, total_clip):
    per_layer = total_clip / np.sqrt(len(layer_names))
    return {k: per_layer for k in layer_names}


clip_weighted = weighted_clip_split(layer_names_dp, grad_norm_ema_dp, TOTAL_CLIP_NORM)
clip_uniform_split = uniform_clip_split(layer_names_dp, TOTAL_CLIP_NORM)

print("Clip norm comparison (uniform-split vs weighted-split):")
for name in layer_names_dp:
    print(f"  {name:20s} uniform_C={clip_uniform_split[name]:.4f}  weighted_C={clip_weighted[name]:.4f}")

**Confirmation of the hypothesis, before any training:** the two recurrent `weight_hh` layers receive clip norms **6-7x smaller** under the weighted split than the uniform split (e.g. 0.041-0.046 vs. a flat 0.289) — closely matching the noise-ratio disparity observed back in section 7. This is mechanistically exactly what the hypothesis predicted.

In [ ]:
print("=== Running LAN-RDP (capped) + SGD + 20% sparsify + WEIGHTED clip split ===")
_, hist_lan_weighted, acc_lan_weighted = run_dp_sgd_fedavg_sparse(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, SGD_LR,
    clip_weighted, lan_capped_sigma_dict, 0.2, "LAN-RDP-WEIGHTEDCLIP", seed=SEED
)

print("\n=== Control: Uniform + SGD + 20% sparsify + WEIGHTED clip split (same clip split) ===")
_, hist_uniform_weighted, acc_uniform_weighted = run_dp_sgd_fedavg_sparse(
    client_train, X_test_global, y_test_global, ch, n_classes,
    N_ROUNDS_SCALED, LOCAL_EPOCHS_SCALED, BATCH_SIZE_DPSGD, SGD_LR,
    clip_weighted, uniform_sigma_dict_scaled, 0.2, "UNIFORM-WEIGHTEDCLIP", seed=SEED
)

pd.DataFrame(hist_lan_weighted).to_csv("dpsgd_lan_weightedclip_history.csv", index=False)
pd.DataFrame(hist_uniform_weighted).to_csv("dpsgd_uniform_weightedclip_history.csv", index=False)

### Result: LAN-RDP's first win, under a properly matched control

The control condition (Uniform allocation, **same** weighted clip split) is essential here: it isolates whether the weighted split is a generic improvement that would lift *any* allocation strategy equally (in which case the relative LAN-RDP vs. Uniform comparison would be unchanged), or whether it specifically interacts with LAN-RDP's already-uneven σ_l.

| Configuration | Peak Accuracy | Final (round 15) Accuracy |
|---|---|---|
| Uniform + SGD + 20% sparsify, **uniform clip** (section 11, original) | 38.19% | 35.20% |
| LAN-RDP(capped) + SGD + 20% sparsify, **uniform clip** (section 12, original) | 29.69% | 19.51% |
| Uniform + SGD + 20% sparsify, **weighted clip** (control, this section) | 34.55% | 31.37% |
| **LAN-RDP(capped) + SGD + 20% sparsify, weighted clip** (this section) | **38.84%** | **37.25%** |


